# LanceDB vector database

In [1]:
import lancedb

# i python såmåste det vara Path()
db = lancedb.connect(uri="vector_database")
db

LanceDBConnection(uri='c:\\Skolprojekt\\ai_engineering\\ai_engineering_jonas_gustafsson\\code-alongs\\13_lancedb\\vector_database')

In [2]:
db.uri

'c:\\Skolprojekt\\ai_engineering\\ai_engineering_jonas_gustafsson\\code-alongs\\13_lancedb\\vector_database'

## Read in data

In [37]:
import json

with open("data/animals_text_embeddings.json", "r") as file:
    data = json.loads(file.read())

data

[{'text': 'A small brown dog running.', 'vector': [0.12, 0.85, 0.33]},
 {'text': 'A cat resting quietly on a sofa.', 'vector': [0.4, 0.91, 0.1]},
 {'text': 'A large gray elephant drinking water.',
  'vector': [0.88, 0.22, 0.55]},
 {'text': 'A fast cheetah sprinting across the savannah.',
  'vector': [0.95, 0.12, 0.72]},
 {'text': 'A colorful parrot perched on a branch.',
  'vector': [0.25, 0.66, 0.81]},
 {'text': 'A frog sitting on a lily pad.', 'vector': [0.14, 0.44, 0.27]}]

In [4]:
db.create_table("animals", exist_ok=True, data=data)


LanceTable(name='animals', version=2, _conn=LanceDBConnection(uri='c:\\Skolprojekt\\ai_engineering\\ai_engineering_jonas_gustafsson\\code-alongs\\13_lancedb\\vector_database'))

In [5]:
db.list_tables()

ListTablesResponse(tables=['animals'], page_token=None)

In [6]:
db["animals"]

LanceTable(name='animals', version=2, _conn=LanceDBConnection(uri='c:\\Skolprojekt\\ai_engineering\\ai_engineering_jonas_gustafsson\\code-alongs\\13_lancedb\\vector_database'))

In [7]:
db["animals"].head()


pyarrow.Table
text: string
vector: fixed_size_list<item: float>[3]
  child 0, item: float
----
text: [["A small brown dog running.","A cat resting quietly on a sofa.","A large gray elephant drinking water.","A fast cheetah sprinting across the savannah.","A colorful parrot perched on a branch."]]
vector: [[[0.12,0.85,0.33],[0.4,0.91,0.1],[0.88,0.22,0.55],[0.95,0.12,0.72],[0.25,0.66,0.81]]]

In [8]:
df_animals = db["animals"].to_pandas()
df_animals

,text,vector
0,A small brown dog running.,"[0.12, 0.85, 0.33]"
1,A cat resting quietly on a sofa.,"[0.4, 0.91, 0.1]"
2,A large gray elephant drinking water.,"[0.88, 0.22, 0.55]"
3,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]"
4,A colorful parrot perched on a branch.,"[0.25, 0.66, 0.81]"
5,A frog sitting on a lily pad.,"[0.14, 0.44, 0.27]"
6,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
7,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"


In [9]:
df_animals.iloc

## To add more data

In [10]:
more_data = [
    {"text": "A panda eating bamboo peacefully.", "vector": [0.51, 0.37, 0.82]},
    {"text": "A lion roaring loudly on a rock.", "vector": [0.93, 0.18, 0.41]},
]

db["animals"].add(more_data)

AddResult(version=3)

In [11]:
db["animals"].to_pandas()

,text,vector
0,A small brown dog running.,"[0.12, 0.85, 0.33]"
1,A cat resting quietly on a sofa.,"[0.4, 0.91, 0.1]"
2,A large gray elephant drinking water.,"[0.88, 0.22, 0.55]"
3,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]"
4,A colorful parrot perched on a branch.,"[0.25, 0.66, 0.81]"
5,A frog sitting on a lily pad.,"[0.14, 0.44, 0.27]"
6,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
7,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"
8,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
9,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"


## Create empty table
- Create an empty table first, then place in data
- Need to provide schema

In [12]:
from lancedb.pydantic import LanceModel

class EmployeeSchema(LanceModel):
    first_name: str
    last_name: str
    salary: int

db.create_table(name = "employees",schema=EmployeeSchema, exist_ok=True)

LanceTable(name='employees', version=1, _conn=LanceDBConnection(uri='c:\\Skolprojekt\\ai_engineering\\ai_engineering_jonas_gustafsson\\code-alongs\\13_lancedb\\vector_database'))

In [13]:
data = [{"first_name": "Bibbi", "last_name": "Babblarna", "salary": 1000}]
db["employees"].add(data)

AddResult(version=2)

In [14]:
db["employees"].to_pandas()

,first_name,last_name,salary
0,Bibbi,Babblarna,1000


In [15]:
db.list_tables()

ListTablesResponse(tables=['animals', 'employees'], page_token=None)

In [16]:
db.drop_table("employees")

In [17]:
db.list_tables()


ListTablesResponse(tables=['animals'], page_token=None)

# Vector search
ANN - approximate nearest neighbour for a vector search

1. send in a quesry vector directly and search  
    - this requires that we embed out quesry first using same embedding as what was used in the knowledge base
2. send in a text and let lancedb automaticly embed in and search


In [18]:
db["animals"].to_pandas()

,text,vector
0,A small brown dog running.,"[0.12, 0.85, 0.33]"
1,A cat resting quietly on a sofa.,"[0.4, 0.91, 0.1]"
2,A large gray elephant drinking water.,"[0.88, 0.22, 0.55]"
3,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]"
4,A colorful parrot perched on a branch.,"[0.25, 0.66, 0.81]"
5,A frog sitting on a lily pad.,"[0.14, 0.44, 0.27]"
6,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
7,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"
8,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
9,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"


In [19]:
# asume that we embed our question using same embedding model as the one for animals
# question about elephant
query_vector = [0.9, 0.2, 0.5]
db["animals"].search(query_vector).limit(4).to_pandas()


,text,vector,_distance
0,A large gray elephant drinking water.,"[0.88, 0.22, 0.55]",0.0033
1,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]",0.0094
2,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]",0.0094
3,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]",0.0573


## Embeddings API

- let lancedb embed our documents automatically
- let lancedg embed out quesry automatically and search using natural language

In [23]:
from lancedb.pydantic import Vector
from lancedb.embeddings import get_registry
from dotenv import load_dotenv

load_dotenv()

model = get_registry().get("gemini-text").create(name="gemini-embedding-001")
model

GeminiText(max_retries=7, name='gemini-embedding-001', query_task_type='retrieval_query', source_task_type='retrieval_document')

In [ ]:
embeddings = model.generate_embeddings("Why are databases good at relationships? Because they are relational")


[[-0.024318507,
  0.021380683,
  0.0118576065,
  -0.058168266,
  -0.028883407,
  0.011489711,
  -0.0022447465,
  0.014321048,
  0.011712493,
  -0.00023437424,
  0.012327072,
  -0.014833164,
  -0.0014773919,
  0.034879602,
  0.098589525,
  0.00044369575,
  0.013094193,
  0.007379488,
  0.006603937,
  0.005444807,
  0.013081205,
  -0.01289585,
  0.013336445,
  -0.0007170391,
  0.010834224,
  0.0009772424,
  0.006912347,
  -0.003631615,
  0.05108864,
  0.00036124472,
  0.0036054675,
  0.013380892,
  0.016712347,
  -0.0009248801,
  -0.004533899,
  0.0014884515,
  0.011677108,
  0.0037706646,
  -0.008544303,
  0.0026787387,
  0.00046600803,
  -0.006032636,
  0.029163595,
  -0.006803228,
  -0.011050768,
  0.0052175187,
  0.0064113275,
  -0.0076469267,
  -0.014462639,
  0.029349009,
  0.008311669,
  -0.025334226,
  -0.013655366,
  -0.14455642,
  0.022412555,
  -0.0067298426,
  -0.026344718,
  -0.01991177,
  -0.009647511,
  -0.0030755263,
  0.0016011818,
  -0.0054774648,
  -0.01269149,
  -0.00

In [28]:
import numpy as np
np.array(embeddings).shape

(68, 3072)

In [29]:
class JokeModel(LanceModel):
    joke: str = model.SourceField() #Input to embedding function
    embedding: Vector(3072) = model.VectorField() #Computed embedding in this column

db.create_table("jokes", schema=JokeModel, exist_ok=True)

LanceTable(name='jokes', version=1, _conn=LanceDBConnection(uri='c:\\Skolprojekt\\ai_engineering\\ai_engineering_jonas_gustafsson\\code-alongs\\13_lancedb\\vector_database'))

In [35]:
import pandas as pd

with open("data/jokes.json", "r", encoding="utf-8") as file:
    jokes_data = json.loads(file.read())
